# Jet Engine RUL Using NASA CMAPSS Data

Data URL: https://data.nasa.gov/dataset/cmapss-jet-engine-simulated-data

# Tasks for week 1

1. Business value - Jheffer
    - How the business value relates to false positives/negatives (false positive for example could mean that an engine is serviced when it doesn't need to be)
    - Document this so that we can utilize it for the final presentation
    - Useful information can be found in section 2 of the PDF "Damage Propagation Modelling" included with the use case data
    - What does the model need to provide to actually bring value to the business?

2. Data visualization x2 - Jesper & Linus
    - Visualize the data through graphs and reflect over possible takeaways
    - Correlation map
    - Clustering operational conditions (lower prio)
    - Find patterns in the data (scatter plot)

3. Data preparation - Chris
    - Come up with possible data preparation methods that could be relevant for us and try them out
    - Reflect over motivation to how/why they could be useful
    - Remove null data
    - Clustering operational conditions
    - Compression

# Business Value



In [ ]:
# Install Required packages
%pip install matplotlib
%pip install pandas
%pip install scikit-learn

# Data Gathering

## Data Loading
Data is split into training and testing sets. Let's start by loading the training data.

In [ ]:
import pandas as pd

df_train_FD001 = pd.read_csv("/home/jovyan/work/jet_engine_rul/Data/train_FD001.txt", sep=' ', header=None)
df_train_FD001

In [ ]:
df_train_FD002 = pd.read_csv("/home/jovyan/work/jet_engine_rul/Data/train_FD002.txt", sep=' ', header=None)
df_train_FD002

In [ ]:
df_train_FD003 = pd.read_csv("/home/jovyan/work/jet_engine_rul/Data/train_FD003.txt", sep=' ', header=None)
df_train_FD003

In [ ]:
df_train_FD004 = pd.read_csv("/home/jovyan/work/jet_engine_rul/Data/train_FD004.txt", sep=' ', header=None)
df_train_FD004

# Data Preparation

## Removing null data
The data has columns 26 and 27 which should not be present. Inspecting these columns, it looks like they are the result of training spaces in the data.

In [ ]:
df_train_FD001[(df_train_FD001[26].notna())|(df_train_FD001[27].notna())]

In [ ]:
df_train_FD002[(df_train_FD002[26].notna())|(df_train_FD002[27].notna())]

In [ ]:
df_train_FD003[(df_train_FD003[26].notna())|(df_train_FD003[27].notna())]

In [ ]:
df_train_FD004[(df_train_FD004[26].notna())|(df_train_FD004[27].notna())]

So we can drop columns 26 and 27 from the data sets

In [ ]:
df_train_FD001 = df_train_FD001.iloc[:, :26]
df_train_FD002 = df_train_FD002.iloc[:, :26]
df_train_FD003 = df_train_FD003.iloc[:, :26]
df_train_FD004 = df_train_FD004.iloc[:, :26]

Number of null values per column:

In [ ]:
df_train_FD001.isnull().sum().sum()

In [ ]:
df_train_FD002.isnull().sum().sum()

In [ ]:
df_train_FD003.isnull().sum().sum()

In [ ]:
df_train_FD004.isnull().sum().sum()

## Defining Columns
The first two column names are specified in the readme.txt file attached to the CMAPSS data: Unit Number and Time in Cycles.

The paper titled "Damage Propagation Modelling" attached to the CMAPSS data specifies the operational parameters as:
1. Altitude (0-42K ft.)
2. Mach number (0-0.84)
3. Throttle resolver angle (TRA) (20-100)

Looking at the min and max values in the dataframe description above, we see that these parameters map roughly to columns 2, 3 and 4, and in the same order.

For now we will assume that the sensor data is given in the same order as the specified in "Damage Propagation Modelling".

In [ ]:
df_train_FD001.columns = ['Unit Number', 'Time (Cycles)', 'Altitude', 'Mach Number', 'TRA', "T2", "T24", "T30", "T50", "P2", "P15", "P30", "Nf", "Nc", "epr", "Ps30", "phi", "NRf", "NRc", "BPR", "farB", "htBleed", "Nf_dmd", "PCNfR_dmd", "W31", "W32"]
df_train_FD002.columns = ['Unit Number', 'Time (Cycles)', 'Altitude', 'Mach Number', 'TRA', "T2", "T24", "T30", "T50", "P2", "P15", "P30", "Nf", "Nc", "epr", "Ps30", "phi", "NRf", "NRc", "BPR", "farB", "htBleed", "Nf_dmd", "PCNfR_dmd", "W31", "W32"]
df_train_FD003.columns = ['Unit Number', 'Time (Cycles)', 'Altitude', 'Mach Number', 'TRA', "T2", "T24", "T30", "T50", "P2", "P15", "P30", "Nf", "Nc", "epr", "Ps30", "phi", "NRf", "NRc", "BPR", "farB", "htBleed", "Nf_dmd", "PCNfR_dmd", "W31", "W32"]
df_train_FD004.columns = ['Unit Number', 'Time (Cycles)', 'Altitude', 'Mach Number', 'TRA', "T2", "T24", "T30", "T50", "P2", "P15", "P30", "Nf", "Nc", "epr", "Ps30", "phi", "NRf", "NRc", "BPR", "farB", "htBleed", "Nf_dmd", "PCNfR_dmd", "W31", "W32"]

## Describe the data
Check the data shape in format (rows, columns):

In [ ]:
df_train_FD001.shape

In [ ]:
df_train_FD002.shape

In [ ]:
df_train_FD003.shape

In [ ]:
df_train_FD004.shape

Describe data columns:

In [ ]:
df_train_FD001.describe()

In [ ]:
df_train_FD002.describe()

In [ ]:
df_train_FD003.describe()

In [ ]:
df_train_FD004.describe()

## Uniquifying Unit Numbers
As we can see, unit numbers are duplicated between the individual data sets. If we can uniquify them, we can merge the data sets into one.

In [ ]:
df_train_FD002["Unit Number"] += df_train_FD001["Unit Number"].max()
df_train_FD003["Unit Number"] += df_train_FD002["Unit Number"].max()
df_train_FD004["Unit Number"] += df_train_FD003["Unit Number"].max()

In [ ]:
df_train_FD001["Unit Number"].min()

In [ ]:
df_train_FD002["Unit Number"].min()

In [ ]:
df_train_FD003["Unit Number"].min()

In [ ]:
df_train_FD004["Unit Number"].min()

In [ ]:
df_train_FD001["Unit Number"].max()

In [ ]:
df_train_FD002["Unit Number"].max()

In [ ]:
df_train_FD003["Unit Number"].max()

In [ ]:
df_train_FD004["Unit Number"].max()

## Merging Data Sets
The data can now be merged into a single data set as we have unique Unit Numbers.

In [ ]:
df_train = pd.concat([df_train_FD001, df_train_FD002, df_train_FD003, df_train_FD004], axis=0)
df_train

In [ ]:
df_train.describe()

## Clustering the operational conditions

Data is given for six operational conditions determined by the Altitude, Mach Number and TRA columns. Let's see if we can cluster these.

In [ ]:
# Get the operational data columns:
df_conditions = df_train[["Altitude", "Mach Number", "TRA"]]
df_conditions

Next we need to scale the data for KMeans:

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
df_conditions_scaled = scaler.fit_transform(df_conditions)

Now we can perform a KMeans fit:

In [ ]:
from sklearn.cluster import KMeans
num_operational_conditions = 6 # The number of operational conditions are given.
km = KMeans(n_clusters=num_operational_conditions)
df_train["Conditions"] = km.fit_predict(df_conditions_scaled)
df_train

Now we can plot the clusters:

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

# Define colors for 6 clusters
colors = ['#e41a1c', '#377eb8', '#4daf4a', '#984ea3', '#ff7f00', '#ffff33']

for cluster in range(num_operational_conditions):
    cluster_data = df_train[df_train['Conditions'] == cluster]
    
    ax.scatter(
        cluster_data["Altitude"],
        cluster_data["Mach Number"],
        cluster_data["TRA"],
        c=colors[cluster],
        label=f'Condition {cluster}',
        s=50,
        alpha=0.8
    )

ax.set_xlabel("Altitude (x1000 ft)")
ax.set_ylabel("Mach Number")
ax.set_zlabel("TRA")
ax.set_title("Operational Condition Clusters")
ax.legend()
plt.show()

The clustering is very well separated. We can evaluate it anyway. Values greater than 0.7 indicates excellent clustering.

In [ ]:
from sklearn.metrics import silhouette_score
silhouette_score(df_conditions_scaled, df_train['Conditions'], metric="euclidean", sample_size=25000)

## Calculate the RUL for each unit

Get the times at which each unit fails:

In [ ]:
df_train["Fail Time"] = df_train.groupby("Unit Number")["Time (Cycles)"].transform('max')

Now we can use these fail times to calculate the RUL for each unit:

In [ ]:
df_train["RUL"] = df_train["Fail Time"] - df_train["Time (Cycles)"]
df_train